### Hybrid Search Langchain

In [4]:
!pip install --upgrade --quiet  pinecone-client pinecone-text pinecone-notebooks

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

api_key = os.getenv('PINECONE_API_KEY')

In [2]:
from langchain_community.retrievers import PineconeHybridSearchRetriever

In [3]:
import os 
from pinecone import Pinecone, ServerlessSpec
index_name = 'hybrid-search-langchain-pinecone'

# initilize the pinecone client
pc = Pinecone(api_key=api_key)

if index_name is not pc.list_indexes().names():
    pc.create_index(
        name = index_name,
        dimension = 384, # Dimension of dense vector
        metric = 'dotproduct', # Sparse values supported only for dotproduct
        spec = ServerlessSpec(cloud='aws',region='us-east-1')
    )

/Volumes/Sandisk 1TB/Documents/Generative AI Krish Naik/Hybrid Search RAG With Pinecone & Langchain/venv/lib/python3.12/site-packages/pinecone/data/index.py:1: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from tqdm.autonotebook import tqdm


PineconeApiException: (409)
Reason: Conflict
HTTP response headers: HTTPHeaderDict({'content-type': 'text/plain; charset=utf-8', 'vary': 'origin, access-control-request-method, access-control-request-headers', 'access-control-allow-origin': '*', 'access-control-expose-headers': '*', 'x-pinecone-api-version': '2024-07', 'x-cloud-trace-context': 'f088e28eed3d55b2911d3cf9f4fb4ec7', 'date': 'Tue, 19 May 2026 07:58:29 GMT', 'server': 'Google Frontend', 'Content-Length': '85', 'Via': '1.1 google', 'Alt-Svc': 'h3=":443"; ma=2592000'})
HTTP response body: {"error":{"code":"ALREADY_EXISTS","message":"Resource  already exists"},"status":409}


In [4]:
index = pc.Index(index_name)
index 

In [6]:
# Vector embedding
import os 
from dotenv import load_dotenv
load_dotenv()

os.environ['HF_TOKEN'] = os.getenv("HF_TOKEN")

from langchain.embeddings import HuggingFaceEmbeddings
embeddings = HuggingFaceEmbeddings(model_name = 'all-MiniLM-L6-v2',show_progress=True)
embeddings

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 8201.80it/s]


HuggingFaceEmbeddings(client=SentenceTransformer(
  (0): Transformer({'transformer_task': 'feature-extraction', 'modality_config': {'text': {'method': 'forward', 'method_output_name': 'last_hidden_state'}}, 'module_output_name': 'token_embeddings', 'architecture': 'BertModel'})
  (1): Pooling({'embedding_dimension': 384, 'pooling_mode': 'mean', 'include_prompt': True})
  (2): Normalize({})
), model_name='all-MiniLM-L6-v2', cache_folder=None, model_kwargs={}, encode_kwargs={}, multi_process=False, show_progress=True)

In [7]:
from pinecone_text.sparse import BM25Encoder
bm25_encoder = BM25Encoder().default()
bm25_encoder

In [18]:
sentences=[
    "In 2023, I visited Paris",
        "In 2022, I visited New York",
        "In 2021, I visited New Orleans",

]

# Applying TF_IDF Values on this sentences
bm25_encoder.fit(sentences)

# Store the values to a json file
bm25_encoder.dump('bm25_values.json')



100%|██████████| 3/3 [00:00<00:00, 1693.98it/s]


In [9]:
retriever = PineconeHybridSearchRetriever(embeddings=embeddings,sparse_encoder=bm25_encoder,index=index)

In [10]:
retriever

PineconeHybridSearchRetriever(embeddings=HuggingFaceEmbeddings(client=SentenceTransformer(
  (0): Transformer({'transformer_task': 'feature-extraction', 'modality_config': {'text': {'method': 'forward', 'method_output_name': 'last_hidden_state'}}, 'module_output_name': 'token_embeddings', 'architecture': 'BertModel'})
  (1): Pooling({'embedding_dimension': 384, 'pooling_mode': 'mean', 'include_prompt': True})
  (2): Normalize({})
), model_name='all-MiniLM-L6-v2', cache_folder=None, model_kwargs={}, encode_kwargs={}, multi_process=False, show_progress=True), sparse_encoder=<pinecone_text.sparse.bm25_encoder.BM25Encoder object at 0x16ae93500>, index=<pinecone.data.index.Index object at 0x1142e6840>)

In [11]:
retriever.add_texts(
    [
    "In 2023, I visited Paris",
        "In 2022, I visited New York",
        "In 2021, I visited New Orleans",

]
)

  0%|          | 0/1 [00:00<?, ?it/s]

100%|██████████| 1/1 [00:03<00:00,  3.28s/it]


In [15]:
retriever.invoke("What city did i visit last")

Batches: 100%|██████████| 1/1 [00:00<00:00, 11.51it/s]


[Document(page_content='In 2021, I visited New Orleans'),
 Document(page_content='In 2022, I visited New York'),
 Document(page_content='In 2023, I visited Paris')]